# Quantizing and Exporting a PyTorch Model for STM32F Deployment

This notebook guides through the process of:
1. Loading a PyTorch KWS-Mamba model
2. Applying int8 quantization (both weights and activations)
3. Exporting the model in a format compatible with STM32F microcontrollers
4. Generating appropriate C header files for embedded deployment
5. Comparing the accuracy between original and quantized models

## 1. Import Required Libraries

In [1]:
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
import torch.quantization
from torch.quantization import quantize_dynamic, QuantStub, DeQuantStub
from collections import OrderedDict
import struct
import copy

# Add parent directory to path to import from src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

# Check if CUDA is available
device = 'cpu'
print(f"Using device: {device}")

Using device: cpu


## 2. Load Model Architecture and Weights

In [2]:
# Import the model architecture
from src.models.model import KeywordSpottingModel_with_cls  # Adjust import based on your model structure
from src.data.config import dataset, data_loader, model as model_config, optimizer as optimizer_config, scheduler as scheduler_config, training

# Path to the trained model weights
MODEL_PATH = 'best_model.pth'  # Adjust path as needed


def load_model(model_path):
    """
    Load the trained model
    """
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    # Assuming checkpoint is directly the state dict
    model = KeywordSpottingModel_with_cls(**model_config)
    model.load_state_dict(checkpoint,strict=False)
    
    model.to(device)
    model.eval()
    return model

try:
    # Try to load the model
    model = load_model(MODEL_PATH)
    print("Model loaded successfully")
    print(model)
except Exception as e:
    print(f"Error loading model: {e}")

Model loaded successfully
KeywordSpottingModel_with_cls(
  (proj): Linear(in_features=69, out_features=136, bias=True)
  (mamba_layers): ModuleList(
    (0-1): 2 x MambaBlock(
      (in_proj): Linear(in_features=136, out_features=544, bias=False)
      (conv1d): Conv1d(272, 272, kernel_size=(10,), stride=(1,), padding=(9,), groups=272)
      (x_proj): Linear(in_features=272, out_features=111, bias=False)
      (dt_proj): Linear(in_features=9, out_features=272, bias=True)
      (out_proj): Linear(in_features=272, out_features=136, bias=False)
    )
  )
  (layer_norms): ModuleList(
    (0-1): 2 x RMSNorm((136,), eps=None, elementwise_affine=True)
  )
  (fc): Linear(in_features=136, out_features=12, bias=True)
  (dropout): Dropout(p=0.134439213335519, inplace=False)
)


## 3. Prepare Model for Quantization

To prepare the model for quantization, we need to modify it to include `QuantStub` and `DeQuantStub` modules, which will insert the quantization and dequantization operations.

In [3]:
class QuantizableKWSModel(nn.Module):
    """
    Wrapper class to make the model quantization-ready
    """
    def __init__(self, model):
        super().__init__()
        self.quant = QuantStub()
        self.model = model
        self.dequant = DeQuantStub()
        
    def forward(self, x):
        x = self.quant(x)
        x = self.model(x)
        x = self.dequant(x)
        return x
    
    def get_original_model(self):
        return self.model

# Create a quantizable version of the model
quantizable_model = QuantizableKWSModel(copy.deepcopy(model))
quantizable_model.eval()

QuantizableKWSModel(
  (quant): QuantStub()
  (model): KeywordSpottingModel_with_cls(
    (proj): Linear(in_features=69, out_features=136, bias=True)
    (mamba_layers): ModuleList(
      (0-1): 2 x MambaBlock(
        (in_proj): Linear(in_features=136, out_features=544, bias=False)
        (conv1d): Conv1d(272, 272, kernel_size=(10,), stride=(1,), padding=(9,), groups=272)
        (x_proj): Linear(in_features=272, out_features=111, bias=False)
        (dt_proj): Linear(in_features=9, out_features=272, bias=True)
        (out_proj): Linear(in_features=272, out_features=136, bias=False)
      )
    )
    (layer_norms): ModuleList(
      (0-1): 2 x RMSNorm((136,), eps=None, elementwise_affine=True)
    )
    (fc): Linear(in_features=136, out_features=12, bias=True)
    (dropout): Dropout(p=0.134439213335519, inplace=False)
  )
  (dequant): DeQuantStub()
)

## 4. Apply Dynamic Quantization

We'll use dynamic quantization, which quantizes the weights to int8 but keeps activations in floating point during computation and quantizes them on-the-fly.

In [4]:
def apply_dynamic_quantization(model):
    """
    Apply dynamic quantization to the model
    """
    quantized_model = torch.quantization.quantize_dynamic(
        model, 
        {nn.Linear, nn.Conv1d}, 
        dtype=torch.qint8
    )
    return quantized_model

# Apply dynamic quantization
quantized_model = apply_dynamic_quantization(quantizable_model)
print("Model quantized successfully")

Model quantized successfully


## 5. Static Quantization Alternative

For better performance on STM32, we can also use static quantization, which requires a calibration step with representative data.

In [5]:
# # Load a small subset of data for calibration
# try:
#     from src.data.data_loader import get_dataloader
    
#     def get_calibration_data(num_samples=100):
#         """
#         Get a small batch of data for calibration
#         """
#         # Adjust parameters according to your data loader
#         dataloader = get_dataloader(batch_size=num_samples, split='val')
#         for batch in dataloader:
#             inputs, _ = batch
#             return inputs.to(device)
    
#     calibration_data = get_calibration_data()
#     print(f"Loaded {calibration_data.shape[0]} samples for calibration")
    
#     # Create a new model for static quantization
#     static_quantizable_model = QuantizableKWSModel(copy.deepcopy(model))
#     static_quantizable_model.eval()
    
#     # Specify quantization configuration
#     static_quantizable_model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
    
#     # Prepare model for static quantization
#     torch.quantization.prepare(static_quantizable_model, inplace=True)
    
#     # Calibrate with data
#     with torch.no_grad():
#         static_quantizable_model(calibration_data)
    
#     # Convert to quantized model
#     static_quantized_model = torch.quantization.convert(static_quantizable_model, inplace=True)
#     print("Static quantization completed successfully")
    
# except Exception as e:
#     print(f"Could not perform static quantization: {e}")
#     print("Continuing with dynamic quantization only")
#     static_quantized_model = None

## 6. Evaluate Model Accuracy

Compare the accuracy of the original and quantized models.

In [6]:
from src.data.data_loader import load_speech_commands_dataset, TFDatasetAdapter, load_bg_noise_dataset
from torch.utils.data import DataLoader


2025-04-21 11:22:31.516745: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-21 11:22:31.735571: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-21 11:22:31.780613: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-21 11:22:32.562437: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Co

In [10]:
def evaluate_model(model_name, model, device='cpu'):
    """
    Evaluate the model on the test set
    """
    try:
        import torch.nn.functional as F
        import types  # Import types for monkey patching
        
        model.to(device)
        model.eval()
        
        # Fix for MambaBlock in quantized models
        if hasattr(model, 'model') and hasattr(model.model, 'mamba_layers'):
            for layer_idx, layer in enumerate(model.model.mamba_layers):
                # Store original forward method
                if not hasattr(layer, '_original_forward'):
                    layer._original_forward = layer.forward
                    
                # Define safe forward method as a workaround for @ operator issues
                def safe_forward(self, x):
                    try:
                        return self._original_forward(x)
                    except TypeError as e:
                        if "unsupported operand type(s) for @" in str(e):
                            print(f"Converting tensor to float to handle @ operation")
                            # Convert tensors to float for matrix multiplication then back to original dtype
                            x_float = x.float()
                            result = self._original_forward(x_float)
                            return result.to(x.dtype) if hasattr(x, 'dtype') else result
                        raise
                
                # Apply the monkey-patched forward method
                layer.forward = types.MethodType(safe_forward, layer)
        
        train_ds, val_ds, test_ds, silence_ds, info = load_speech_commands_dataset(reduced=True)
        pytorch_test_dataset = TFDatasetAdapter(test_ds, None, **dataset, augmentation=None)

        test_loader = DataLoader(pytorch_test_dataset, **data_loader, shuffle=False)
        
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                try:
                    outputs = model(inputs)
                except TypeError as e:
                    if "unsupported operand type(s) for @" in str(e):
                        print("Falling back to float conversion for inputs")
                        outputs = model(inputs.float())
                    else:
                        raise
                
                _, predicted = torch.max(outputs, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        accuracy = 100 * correct / total
        print(f"{model_name} Accuracy: {accuracy:.2f}%")
        return accuracy
        
    except Exception as e:
        print(f"Error during evaluation: {e}")
        import traceback
        traceback.print_exc()
        return None

# Evaluate the original model
print("Evaluating models...")
original_accuracy = evaluate_model("Original", model, device)

# Evaluate the dynamically quantized model
dyn_quantized_accuracy = evaluate_model("Dynamic Quantized", quantized_model, device)

# Print comparison
if original_accuracy and dyn_quantized_accuracy:
    print(f"Accuracy difference (Dynamic): {original_accuracy - dyn_quantized_accuracy:.2f}%")

Evaluating models...
Original Accuracy: 47.52%
Original Accuracy: 47.52%
Converting tensor to float to handle @ operation
Converting tensor to float to handle @ operation
Falling back to float conversion for inputs
Falling back to float conversion for inputs
Converting tensor to float to handle @ operation
Converting tensor to float to handle @ operation
Error during evaluation: unsupported operand type(s) for @: 'method' and 'Tensor'
Error during evaluation: unsupported operand type(s) for @: 'method' and 'Tensor'


Traceback (most recent call last):
  File "/tmp/ipykernel_105990/3060697379.py", line 19, in safe_forward
    return original_forward(x)
  File "/usr/local/lib/python3.10/dist-packages/mambapy/mamba.py", line 222, in forward
    y = self.ssm(x, z)
  File "/usr/local/lib/python3.10/dist-packages/mambapy/mamba.py", line 247, in ssm
    delta = self.dt_proj.weight @ delta.transpose(1, 2) # (ED, dt_rank) @ (B, L, dt_rank) -> (B, ED, L)
TypeError: unsupported operand type(s) for @: 'method' and 'Tensor'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_105990/3218978908.py", line 22, in safe_forward
    return self._original_forward(x)
  File "/tmp/ipykernel_105990/3060697379.py", line 23, in safe_forward
    return original_forward(x.float()).to(x.dtype)
  File "/usr/local/lib/python3.10/dist-packages/mambapy/mamba.py", line 222, in forward
    y = self.ssm(x, z)
  File "/usr/local/lib/python3.10/dist-packages/ma

## 7. Export Model for STM32

Now we'll export the quantized model to a format compatible with STM32F microcontrollers.

In [8]:
def extract_layer_info(module, layer_name='', info=None, prefix=''):
    """
    Extract weights, biases and other parameters from model layers
    """
    if info is None:
        info = {}
        
    for name, child in module.named_children():
        child_name = prefix + '.' + name if prefix else name
        
        if isinstance(child, (nn.quantized.LinearPackedParams, nn.quantized.Conv1dPackedParams)):
            # Handle packed parameters for quantized layers
            info[child_name] = {
                'type': child.__class__.__name__,
                'params': child._packed_params
            }
        elif isinstance(child, nn.quantized.Linear) or isinstance(child, nn.quantized.Conv1d):
            # Extract quantized weight and bias information
            info[child_name] = {
                'type': child.__class__.__name__,
                'weight': child.weight(),
                'bias': child.bias() if child.bias() is not None else None,
                'scale': child.scale,
                'zero_point': child.zero_point
            }
            
            # Add additional parameters for Conv1d
            if isinstance(child, nn.quantized.Conv1d):
                info[child_name].update({
                    'kernel_size': child.kernel_size,
                    'stride': child.stride,
                    'padding': child.padding,
                    'groups': child.groups
                })
        elif hasattr(child, 'weight') and isinstance(child.weight, torch.Tensor):
            # Handle regular layer with weight parameter
            bias = child.bias if hasattr(child, 'bias') and child.bias is not None else None
            info[child_name] = {
                'type': child.__class__.__name__,
                'weight': child.weight,
                'bias': bias
            }
            
            # Add parameters for specific layer types
            if isinstance(child, nn.Conv1d):
                info[child_name].update({
                    'in_channels': child.in_channels,
                    'out_channels': child.out_channels,
                    'kernel_size': child.kernel_size,
                    'stride': child.stride,
                    'padding': child.padding,
                    'groups': child.groups
                })
            elif isinstance(child, nn.Linear):
                info[child_name].update({
                    'in_features': child.in_features,
                    'out_features': child.out_features
                })
        
        # Recursively process nested modules
        extract_layer_info(child, child_name, info, child_name)
    
    return info

# Choose which quantized model to export
model_to_export = quantized_model
if static_quantized_model is not None:
    if static_quantized_accuracy and dyn_quantized_accuracy and static_quantized_accuracy > dyn_quantized_accuracy:
        model_to_export = static_quantized_model
        print("Using static quantized model for export (better accuracy)")
    else:
        print("Using dynamic quantized model for export")
        
# Extract layer information from the chosen model
print("Extracting layer information...")
layer_info = extract_layer_info(model_to_export)

NameError: name 'static_quantized_model' is not defined

## 8. Create Model Architecture JSON

Generate a JSON file describing the model architecture.

In [ ]:
def tensor_to_dict(tensor):
    """
    Convert tensor to a serializable dict with shape and data
    """
    if tensor is None:
        return None
    
    # For quantized tensors
    if hasattr(tensor, 'int_repr') and callable(tensor.int_repr):
        int_data = tensor.int_repr().cpu().numpy().tolist()
        return {
            'shape': list(tensor.shape),
            'data_type': 'int8',
            'scale': float(tensor.q_scale()),
            'zero_point': int(tensor.q_zero_point()),
            'data': int_data
        }
    
    # For regular tensors
    return {
        'shape': list(tensor.shape),
        'data_type': str(tensor.dtype).split('.')[-1],
        'data': tensor.detach().cpu().numpy().tolist()
    }

def create_architecture_json(model_info):
    """
    Create a JSON file describing the model architecture
    """
    architecture = {
        'layers': []
    }
    
    for layer_name, info in model_info.items():
        layer_type = info.get('type')
        
        # Skip the quant and dequant stubs
        if 'quant' in layer_name.lower() or 'dequant' in layer_name.lower():
            continue
            
        layer_dict = {'name': layer_name, 'type': layer_type}
        
        # Add layer-specific configuration
        if 'Linear' in layer_type:
            if 'in_features' in info:
                layer_dict['in_features'] = info['in_features']
                layer_dict['out_features'] = info['out_features']
            
        elif 'Conv' in layer_type:
            if 'in_channels' in info:
                layer_dict['in_channels'] = info['in_channels']
                layer_dict['out_channels'] = info['out_channels']
            layer_dict['kernel_size'] = info.get('kernel_size', [1])
            layer_dict['stride'] = info.get('stride', [1])
            layer_dict['padding'] = info.get('padding', [0])
            layer_dict['groups'] = info.get('groups', 1)
        
        # Add quantization info if available
        if 'scale' in info:
            layer_dict['scale'] = float(info['scale'])
            layer_dict['zero_point'] = int(info['zero_point'])
        
        architecture['layers'].append(layer_dict)
    
    return architecture

# Create and save architecture JSON
architecture = create_architecture_json(layer_info)

# Save to file
with open('model_config.json', 'w') as f:
    json.dump(architecture, f, indent=2)

print("Model architecture saved to model_config.json")

## 9. Generate C Header File for Model Weights

Generate a C header file with the quantized weights for embedded deployment.

In [ ]:
def generate_c_header(model_info, file_path='quantized_model_weights.h'):
    """
    Generate a C header file with model weights and parameters
    """
    header = [
        "/* Auto-generated header file from PyTorch model */",
        "#ifndef QUANTIZED_MODEL_WEIGHTS_H",
        "#define QUANTIZED_MODEL_WEIGHTS_H",
        "",
        "#include <stdint.h>",
        "",
        "/* Model Layer Configuration */",
        "typedef struct {"]
    
    # Define the layer configuration structure
    header.extend([
        "    const char* name;        /* Layer name */",
        "    const char* type;        /* Layer type: Conv1d, Linear, etc. */",
        "    int input_size;          /* Input size (features/channels) */",
        "    int output_size;         /* Output size (features/channels) */",
        "    int kernel_size;         /* Kernel size (for Conv layers) */",
        "    int stride;              /* Stride (for Conv layers) */",
        "    int padding;             /* Padding (for Conv layers) */",
        "    float scale;             /* Scale factor for quantization */",
        "    int8_t zero_point;       /* Zero point for quantization */",
        "    const int8_t* weights;   /* Pointer to weights array */",
        "    const int32_t* bias;     /* Pointer to bias array */",
        "    int weights_size;        /* Size of weights array */",
        "    int bias_size;           /* Size of bias array */"
    ])
    
    header.append("} layer_config_t;")
    header.append("")
    
    # Add weight arrays for each layer
    weight_arrays = []
    bias_arrays = []
    layer_configs = []
    layer_count = 0
    
    for layer_name, info in model_info.items():
        # Skip layers without weights or non-parameter layers
        if 'weight' not in info or 'type' not in info:
            continue
            
        # Skip the quant/dequant stubs
        if 'quant' in layer_name.lower() or 'dequant' in layer_name.lower():
            continue
            
        layer_type = info['type']
        layer_type_clean = layer_type.replace('nn.quantized.', '').replace('nn.', '')
        
        # Create C-friendly layer name
        c_layer_name = layer_name.replace('.', '_').replace('model_', '')
        
        # Process weights
        weight = info['weight']
        if hasattr(weight, 'int_repr') and callable(weight.int_repr):
            # Quantized tensor
            weight_data = weight.int_repr().cpu().numpy().flatten().astype(np.int8)
        else:
            # Regular tensor - quantize to int8
            weight_np = weight.detach().cpu().numpy()
            
            # Compute scale and zero point for quantization
            weight_min = weight_np.min()
            weight_max = weight_np.max()
            weight_scale = (weight_max - weight_min) / 255.0
            weight_zero_point = -round(weight_min / weight_scale)
            weight_zero_point = max(0, min(255, weight_zero_point))  # Clamp to 0-255
            
            # Apply quantization
            weight_data = np.round(weight_np / weight_scale + weight_zero_point).astype(np.int8)
        
        # Process bias if available
        bias = info.get('bias')
        if bias is not None:
            if hasattr(bias, 'int_repr') and callable(bias.int_repr):
                bias_data = bias.int_repr().cpu().numpy().flatten().astype(np.int32)
            else:
                bias_np = bias.detach().cpu().numpy()
                # For bias, we use int32 for higher precision
                bias_scale = weight_scale * info.get('scale', 1.0)  # Combine weight and activation scale
                bias_data = np.round(bias_np / bias_scale).astype(np.int32)
        else:
            bias_data = np.array([], dtype=np.int32)
        
        # Weight array
        weight_array_name = f"LAYER_{c_layer_name.upper()}_WEIGHTS"
        weight_array = [f"/* Weights for {layer_name} */"]
        weight_array.append(f"static const int8_t {weight_array_name}[] = {{")
        
        # Format weights in rows of 16 values
        for i in range(0, len(weight_data), 16):
            row = weight_data[i:i+16]
            weight_array.append("    " + ", ".join([f"{int(val)}" for val in row]) + ",")
        
        weight_array[-1] = weight_array[-1].rstrip(",")  # Remove trailing comma
        weight_array.append("};")
        weight_array.append("")
        weight_arrays.extend(weight_array)
        
        # Bias array if available
        if len(bias_data) > 0:
            bias_array_name = f"LAYER_{c_layer_name.upper()}_BIAS"
            bias_array = [f"/* Bias for {layer_name} */"]
            bias_array.append(f"static const int32_t {bias_array_name}[] = {{")
            
            # Format bias in rows of 8 values
            for i in range(0, len(bias_data), 8):
                row = bias_data[i:i+8]
                bias_array.append("    " + ", ".join([f"{int(val)}" for val in row]) + ",")
            
            bias_array[-1] = bias_array[-1].rstrip(",")  # Remove trailing comma
            bias_array.append("};")
            bias_array.append("")
            bias_arrays.extend(bias_array)
        else:
            bias_array_name = "NULL"  # No bias
        
        # Get layer dimensions
        if 'Linear' in layer_type:
            in_size = info.get('in_features', weight.shape[1] if len(weight.shape) > 1 else 0)
            out_size = info.get('out_features', weight.shape[0] if len(weight.shape) > 0 else 0)
            kernel_size = 1
            stride = 1
            padding = 0
        elif 'Conv' in layer_type:
            in_size = info.get('in_channels', weight.shape[1] if len(weight.shape) > 1 else 0)
            out_size = info.get('out_channels', weight.shape[0] if len(weight.shape) > 0 else 0)
            if isinstance(info.get('kernel_size'), (list, tuple)):
                kernel_size = info['kernel_size'][0] if info.get('kernel_size') else 1
            else:
                kernel_size = info.get('kernel_size', 1)
            if isinstance(info.get('stride'), (list, tuple)):
                stride = info['stride'][0] if info.get('stride') else 1
            else:
                stride = info.get('stride', 1)
            if isinstance(info.get('padding'), (list, tuple)):
                padding = info['padding'][0] if info.get('padding') else 0
            else:
                padding = info.get('padding', 0)
        else:
            in_size = weight.shape[1] if len(weight.shape) > 1 else 0
            out_size = weight.shape[0] if len(weight.shape) > 0 else 0
            kernel_size = 1
            stride = 1
            padding = 0
        
        # Create layer configuration
        layer_config = f"    /* Layer {layer_count}: {layer_name} */\n"
        layer_config += f"    {{\
"
        layer_config += f"        \"{c_layer_name}\",           /* name */\n"
        layer_config += f"        \"{layer_type_clean}\",       /* type */\n"
        layer_config += f"        {in_size},                /* input_size */\n"
        layer_config += f"        {out_size},               /* output_size */\n"
        layer_config += f"        {kernel_size},                /* kernel_size */\n"
        layer_config += f"        {stride},                /* stride */\n"
        layer_config += f"        {padding},                /* padding */\n"
        layer_config += f"        {info.get('scale', 1.0):.6f}f,         /* scale */\n"
        layer_config += f"        {int(info.get('zero_point', 0))},                /* zero_point */\n"
        layer_config += f"        {weight_array_name},  /* weights */\n"
        layer_config += f"        {bias_array_name},  /* bias */\n"
        layer_config += f"        {len(weight_data)},              /* weights_size */\n"
        layer_config += f"        {len(bias_data)}               /* bias_size */\n"
        layer_config += f"    }}"
        
        layer_configs.append(layer_config)
        layer_count += 1
    
    # Add weights and bias arrays to header
    header.extend(weight_arrays)
    header.extend(bias_arrays)
    
    # Add layer configuration array
    header.append("/* Layer configurations */")
    header.append(f"#define MODEL_NUM_LAYERS {layer_count}")
    header.append("")
    header.append("static const layer_config_t MODEL_LAYERS[MODEL_NUM_LAYERS] = {")
    
    # Add each layer configuration
    for i, config in enumerate(layer_configs):
        header.append(config + ("," if i < len(layer_configs)-1 else ""))
    
    header.append("};")
    header.append("")
    header.append("#endif /* QUANTIZED_MODEL_WEIGHTS_H */")
    
    # Write to file
    with open(file_path, 'w') as f:
        f.write('\n'.join(header))
    
    return file_path

# Generate C header file
header_file = generate_c_header(layer_info)
print(f"Generated C header file: {header_file}")

## 10. Generate a Simple Inference Implementation in C (Optional)

Generate a basic C implementation for inference using the exported model.

In [ ]:
def generate_c_implementation(file_path='kws_model_inference.c'):
    """
    Generate a basic C implementation for inference
    """
    implementation = [
        "/* Auto-generated inference implementation */",
        "#include <stdio.h>",
        "#include <stdlib.h>",
        "#include <string.h>",
        "#include <math.h>",
        "#include "quantized_model_weights.h"",
        "",
        "/* Buffer for intermediate activations - adjust size as needed */",
        "#define MAX_BUFFER_SIZE 32768",
        "static int8_t activation_buffer_1[MAX_BUFFER_SIZE];",
        "static int8_t activation_buffer_2[MAX_BUFFER_SIZE];",
        "",
        "/* Forward declaration of layer functions */",
        "void apply_linear_layer(const layer_config_t *layer_config, ",
        "                      const int8_t *input, int8_t *output, ",
        "                      int input_length);",
        "void apply_conv1d_layer(const layer_config_t *layer_config, ",
        "                      const int8_t *input, int8_t *output, ",
        "                      int input_length, int input_channels);",
        "void apply_relu(int8_t *data, int length);",
        "",
        "/* Main inference function */",
        "void model_inference(const int8_t *input, int input_size, float *output, int output_size) {"]
    
    implementation.extend([
        "    /* We'll use double buffering for activations */",
        "    int8_t *current_input = activation_buffer_1;",
        "    int8_t *current_output = activation_buffer_2;",
        "    int8_t *temp;",
        "",
        "    /* Copy input to activation buffer with proper quantization */",
        "    /* Note: In a real implementation, you would apply input quantization here */",
        "    memcpy(current_input, input, input_size);",
        "",
        "    int current_size = input_size;",
        "    int input_channels = MODEL_LAYERS[0].input_size;",
        "    int input_length = current_size / input_channels;",
        "",
        "    /* Apply each layer */",
        "    for (int i = 0; i < MODEL_NUM_LAYERS; i++) {"]
    )
    
    implementation.extend([
        "        const layer_config_t *layer = &MODEL_LAYERS[i];",
        "",
        "        if (strcmp(layer->type, "Linear") == 0) {"]),
        "            apply_linear_layer(layer, current_input, current_output, input_length);",
        "            input_channels = layer->output_size;",
        "            current_size = input_channels; /* For Linear, output length is 1 */",
        "        } else if (strcmp(layer->type, "Conv1d") == 0) {"]),
        "            apply_conv1d_layer(layer, current_input, current_output, input_length, input_channels);",
        "            input_channels = layer->output_size;",
        "            input_length = (input_length + 2 * layer->padding - layer->kernel_size) / layer->stride + 1;",
        "            current_size = input_length * input_channels;",
        "        }"]),
        "",
        "        apply_relu(current_output, current_size);",
        "",
        "        /* Swap buffers for next layer */",
        "        temp = current_input;",
        "        current_input = current_output;",
        "        current_output = temp;",
        "    }",
        "",
        "    /* Convert final output to float and apply dequantization */",
        "    const layer_config_t *final_layer = &MODEL_LAYERS[MODEL_NUM_LAYERS-1];",
        "    float scale = final_layer->scale;",
        "    int8_t zero_point = final_layer->zero_point;",
        "",
        "    for (int i = 0; i < output_size; i++) {",
        "        output[i] = (float)(current_input[i] - zero_point) * scale;",
        "    }",
        "}",
        "",
        "/* Layer implementation functions */",
        "void apply_linear_layer(const layer_config_t *layer, ",
        "                      const int8_t *input, int8_t *output, ",
        "                      int input_length) {",
        "    const int8_t *weights = layer->weights;",
        "    const int32_t *bias = layer->bias;",
        "    int in_features = layer->input_size;",
        "    int out_features = layer->output_size;",
        "",
        "    /* For simplicity, assume input_length is 1 for linear layers */",
        "    for (int out_idx = 0; out_idx < out_features; out_idx++) {",
        "        int32_t acc = bias ? bias[out_idx] : 0;",
        "        ",
        "        for (int in_idx = 0; in_idx < in_features; in_idx++) {",
        "            acc += (int32_t)input[in_idx] * (int32_t)weights[out_idx * in_features + in_idx];",
        "        }",
        "        ",
        "        /* Requantize to output scale and zero_point */",
        "        /* In a proper implementation, this would use proper quantization math */",
        "        acc = acc / 256;  /* Simple scaling, replace with proper requantization */",
        "        ",
        "        /* Clamp to int8 range */",
        "        if (acc < -128) acc = -128;",
        "        if (acc > 127) acc = 127;",
        "        ",
        "        output[out_idx] = (int8_t)acc;",
        "    }",
        "}",
        "",
        "void apply_conv1d_layer(const layer_config_t *layer, ",
        "                      const int8_t *input, int8_t *output, ",
        "                      int input_length, int input_channels) {",
        "    const int8_t *weights = layer->weights;",
        "    const int32_t *bias = layer->bias;",
        "    int in_channels = layer->input_size;",
        "    int out_channels = layer->output_size;",
        "    int kernel_size = layer->kernel_size;",
        "    int stride = layer->stride;",
        "    int padding = layer->padding;",
        "",
        "    /* Calculate output dimensions */",
        "    int output_length = (input_length + 2 * padding - kernel_size) / stride + 1;",
        "",
        "    /* For each output position and channel */",
        "    for (int out_ch = 0; out_ch < out_channels; out_ch++) {",
        "        for (int out_pos = 0; out_pos < output_length; out_pos++) {",
        "            int output_idx = out_ch * output_length + out_pos;",
        "            int32_t acc = bias ? bias[out_ch] : 0;",
        "            ",
        "            /* For each kernel position and input channel */",
        "            for (int k = 0; k < kernel_size; k++) {",
        "                int in_pos = out_pos * stride + k - padding;",
        "                ",
        "                if (in_pos >= 0 && in_pos < input_length) {",
        "                    for (int in_ch = 0; in_ch < in_channels; in_ch++) {",
        "                        int input_idx = in_ch * input_length + in_pos;",
        "                        int weight_idx = (out_ch * in_channels * kernel_size) + ",
        "                                         (in_ch * kernel_size) + k;",
        "                        ",
        "                        acc += (int32_t)input[input_idx] * (int32_t)weights[weight_idx];",
        "                    }",
        "                }",
        "            }",
        "            ",
        "            /* Requantize (simplified) */",
        "            acc = acc / 256;  /* Simple scaling, replace with proper requantization */",
        "            ",
        "            /* Clamp to int8 range */",
        "            if (acc < -128) acc = -128;",
        "            if (acc > 127) acc = 127;",
        "            ",
        "            output[output_idx] = (int8_t)acc;",
        "        }",
        "    }",
        "}",
        "",
        "void apply_relu(int8_t *data, int length) {",
        "    for (int i = 0; i < length; i++) {",
        "        if (data[i] < 0) data[i] = 0;",
        "    }",
        "}",
        "",
        "/* Example usage */",
        "void example_inference() {",
        "    /* Create dummy input data */",
        "    int8_t input[128] = {0};  /* Replace with actual input preparation */",
        "    ",
        "    /* Output buffer */",
        "    float output[10] = {0};  /* Adjust size based on your model */",
        "    ",
        "    /* Run inference */",
        "    model_inference(input, sizeof(input), output, 10);",
        "    ",
        "    /* Process the output */",
        "    int max_idx = 0;",
        "    float max_val = output[0];",
        "    ",
        "    for (int i = 1; i < 10; i++) {",
        "        if (output[i] > max_val) {",
        "            max_val = output[i];",
        "            max_idx = i;",
        "        }",
        "    }",
        "    ",
        "    printf("Detected keyword: %d\n", max_idx);",
        "}"
    ]
    
    # Write to file
    with open(file_path, 'w') as f:
        f.write('\n'.join(implementation))
    
    return file_path

# Generate C implementation file
implementation_file = generate_c_implementation()
print(f"Generated C implementation file: {implementation_file}")

## 11. Summary and Next Steps

In this notebook, we have successfully:
1. Loaded a PyTorch KWS-Mamba model
2. Applied int8 quantization to both weights and activations
3. Evaluated the accuracy impact of quantization
4. Exported the model architecture to JSON format
5. Generated C header files with quantized weights for embedded deployment
6. Created a simple C inference implementation

### Next Steps
- Integrate the generated C code with STM32CubeIDE or Keil project
- Optimize the inference implementation for the specific STM32F microcontroller
- Implement proper preprocessing of audio input on the microcontroller
- Adjust memory usage based on the specific STM32F model's constraints
- Consider using CMSIS-NN library for optimized neural network operations

### Additional Improvements
- Implement more precise quantization math in the C code
- Add proper memory alignment for optimal performance
- Optimize the convolution and matrix multiplication operations
- Profile the execution time and memory usage on the target device
- Consider pruning the model further to reduce memory footprint